# CRISP-DM Phase 3 — Data Preparation (Starter Prototype)

**Purpose:** Prepare the video-game CSVs for a Plotly/Dash dashboard.

Follow the Phase 3 tasks in order: **Select → Clean → Construct → Integrate → Format → Describe outputs**.

**Important:** This is a *prototype*, not completed evidence. Run all cells against the original CSV datasets, inspect the printed results, and record verified results in `3_DataPreparation_starter.md`. Do not accidentally load the small data-dictionary CSV files as the actual datasets.

**Rules from Phase 2:**
- Keep missing sales as missing (`NaN`), not zero.
- A title can appear on multiple platforms: don't delete those records automatically.
- Validate joins so they don't multiply the number of original records.
- For geographic enrichment, countries can be matched cautiously; cities may have the same name in different countries.

## 0. Setup and load the original data

Put this notebook at the repository root, next to the `data/` folder. Edit `DATA_DIR` if your notebook is in another folder.

In [12]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../../data")
OUTPUT_DIR = DATA_DIR / "prepared"

files = {
    "charts": "vg_charts.csv",
    "developers": "vg_developers.csv",
    "publishers": "vg_publishers.csv",
    "cities": "vg_geo_cities.csv",
    "countries": "vg_geo_countries.csv"
}

raw = {name: pd.read_csv(DATA_DIR / filename) for name, filename in files.items()}
for name, df in raw.items():
    print(f"{name}: {len(df):,} rows, {len(df.columns)} columns")

# Safety check: have we loaded the full game dataset, not the dictionary?
required = {"title", "platform", "genre", "total_sales", "release_date"}
assert required.issubset(raw["charts"].columns), "Wrong charts file or missing columns"

charts: 64,016 rows, 14 columns
developers: 731 rows, 3 columns
publishers: 889 rows, 3 columns
cities: 23,412 rows, 3 columns
countries: 262 rows, 5 columns


## 1. Select Data — Rationale for inclusion/exclusion

Start by keeping all 14 original game fields. Data volume is manageable, and `img`/`last_update` may remain useful as optional metadata. We will **not** drop games just because sales or dates are missing. The city lookup is loaded, but *not joined automatically*, since matching on city names alone is unreliable.

**Checkpoint:** The number of rows must stay the same.

In [13]:
game_columns = [
    "img", "title", "platform", "genre", "publisher", "developer",
    "critic_score", "total_sales", "na_sales", "jp_sales",
    "pal_sales", "other_sales", "release_date", "last_update"
]

games = raw["charts"][game_columns].copy()
games["record_id"] = range(1, len(games) + 1)  # stable ID for each original row

print("Selected game rows:", len(games))
print("Missing total sales:", games["total_sales"].isna().sum())
assert len(games) == len(raw["charts"])

# read each row with missing total sales, give head(10)
# games[games["total_sales"].isna()].head(10)

# check which rows have no total sales, but have na, jp or pal sales
# games[(games["total_sales"].isna()) & (games["na_sales"].notnull() | games["jp_sales"].notnull() | games["pal_sales"].notnull())].head(10)

Selected game rows: 64016
Missing total sales: 45094


## 2. Clean Data — Data Cleaning Report

- Strip surrounding spaces in names, but preserve the original capitalization of display labels.
- Parse dates using the source's `YYYY-MM-DD` format.
- Ensure sales and scores are numeric, without replacing missing values with zero.
- *Flag* repeated `title` + `platform` pairs for investigation; don't automatically remove them.

**Checkpoint:** Look at failed conversions, duplicate combinations, and missingness before deciding on further cleaning.

In [14]:
text_columns = ["title", "platform", "genre", "publisher", "developer"]
for column in text_columns:
    games[column] = games[column].astype("string").str.strip()
    games[column] = games[column].replace("", pd.NA)

date_columns = ["release_date", "last_update"]
for column in date_columns:
    before = games[column].notna().sum()
    games[column] = pd.to_datetime(games[column], format="%Y-%m-%d", errors="coerce")
    failed = before - games[column].notna().sum()
    print(f"{column}: {failed} failed date conversions")

sales_columns = ["total_sales", "na_sales", "jp_sales", "pal_sales", "other_sales"]
for column in sales_columns + ["critic_score"]:
    before = games[column].notna().sum()
    games[column] = pd.to_numeric(games[column], errors="coerce")
    failed = before - games[column].notna().sum()
    print(f"{column}: {failed} failed numeric conversions")

print("Repeated title-platform pairs:", games.duplicated(["title", "platform"]).sum())
print("Missing total sales after cleaning:", games["total_sales"].isna().sum())


release_date: 0 failed date conversions
last_update: 0 failed date conversions
total_sales: 0 failed numeric conversions
na_sales: 0 failed numeric conversions
jp_sales: 0 failed numeric conversions
pal_sales: 0 failed numeric conversions
other_sales: 0 failed numeric conversions
critic_score: 0 failed numeric conversions
Repeated title-platform pairs: 225
Missing total sales after cleaning: 45094


## 3. Construct Data — Derived attributes and generated records

Create `release_year` for time-series charts and `has_total_sales` for coverage calculations. `record_id` identifies original rows; it is **not** a unique game-title ID. No entirely new games have been created.

In [15]:
games["release_year"] = games["release_date"].dt.year.astype("Int64")
games["has_total_sales"] = games["total_sales"].notna()

print(games[["title", "platform", "release_year", "total_sales", "has_total_sales"]].head())
print("Rows with a release year:", games["release_year"].notna().sum())
assert games["record_id"].is_unique

                         title platform  release_year  total_sales  \
0           Grand Theft Auto V      PS3          2013        20.32   
1           Grand Theft Auto V      PS4          2014        19.39   
2  Grand Theft Auto: Vice City      PS2          2002        16.15   
3           Grand Theft Auto V     X360          2013        15.86   
4    Call of Duty: Black Ops 3      PS4          2015        15.09   

   has_total_sales  
0             True  
1             True  
2             True  
3             True  
4             True  
Rows with a release year: 56965


## 4. Integrate Data — Add company headquarters (optional enrichment)

Join the developer and publisher reference tables using normalized helper keys. Some names are unmatched; keep those game rows via **left joins**. Keys occurring multiple times in a reference are **excluded pending review**, rather than allowing an ambiguous join. We do not infer company headquarters from sales regions.

In [16]:
# Create temporary keys for matching company names.
games["developer_key"] = games["developer"].str.casefold()
games["publisher_key"] = games["publisher"].str.casefold()

developers = raw["developers"][["developer", "city", "country"]].copy()
developers["developer_key"] = developers["developer"].astype("string").str.strip().str.casefold().replace("", pd.NA)
ambiguous_dev = developers["developer_key"].duplicated(keep=False)
print("Ambiguous developer reference rows:", ambiguous_dev.sum())
developers = developers.loc[~ambiguous_dev & developers["developer_key"].notna(),
                            ["developer_key", "city", "country"]]
developers = developers.rename(columns={"city": "developer_city", "country": "developer_country"})

publishers = raw["publishers"][["publisher", "city", "country"]].copy()
publishers["publisher_key"] = publishers["publisher"].astype("string").str.strip().str.casefold().replace("", pd.NA)
ambiguous_pub = publishers["publisher_key"].duplicated(keep=False)
print("Ambiguous publisher reference rows:", ambiguous_pub.sum())
publishers = publishers.loc[~ambiguous_pub & publishers["publisher_key"].notna(),
                            ["publisher_key", "city", "country"]]
publishers = publishers.rename(columns={"city": "publisher_city", "country": "publisher_country"})

# validate='m:1' prevents a game row matching multiple reference rows.
enriched = games.merge(developers, on="developer_key", how="left", validate="m:1")
enriched = enriched.merge(publishers, on="publisher_key", how="left", validate="m:1")
assert len(enriched) == len(games), "Unexpected row count after company joins"
print("Games matched to a developer headquarters:", enriched["developer_country"].notna().sum())
print("Games matched to a publisher headquarters:", enriched["publisher_country"].notna().sum())


Ambiguous developer reference rows: 0
Ambiguous publisher reference rows: 2
Games matched to a developer headquarters: 18515
Games matched to a publisher headquarters: 31596


### Optional country coordinates

Use the country lookup for company headquarters **only**, not for sales regions. The country lookup stores coordinate text with quotes that must be removed before conversion. If country names don't match exactly (e.g., alternative spellings), keep locations unknown and inspect them later. We deliberately do not join `vg_geo_cities.csv` by city alone, because city names aren't globally unique.

In [17]:
countries = raw["countries"].copy()
countries["country_key"] = countries["Country"].astype("string").str.strip().str.casefold().replace("", pd.NA)

for column in ["Latitude", "Longitude"]:
    countries[column] = pd.to_numeric(
        countries[column].astype("string").str.strip().str.strip('"').str.strip(),
        errors="coerce"
    )

countries["iso3"] = countries["Alpha-3 code"].astype("string").str.strip().str.strip('"')
ambiguous_countries = countries["country_key"].duplicated(keep=False)
print("Ambiguous country reference rows:", ambiguous_countries.sum())
country_ref = countries.loc[~ambiguous_countries & countries["country_key"].notna(),
                            ["country_key", "iso3", "Latitude", "Longitude"]]

for prefix in ["developer", "publisher"]:
    enriched["country_key"] = enriched[f"{prefix}_country"].astype("string").str.strip().str.casefold()
    lookup = country_ref.rename(columns={
        "iso3": f"{prefix}_iso3", "Latitude": f"{prefix}_latitude",
        "Longitude": f"{prefix}_longitude"
    })
    enriched = enriched.merge(lookup, on="country_key", how="left", validate="m:1")
    enriched = enriched.drop(columns="country_key")

assert len(enriched) == len(games)
print("Games with publisher country coordinates:", enriched["publisher_latitude"].notna().sum())

Ambiguous country reference rows: 0
Games with publisher country coordinates: 31343


### Reshape regional sales into long format

The source has one column per region. `melt` creates four rows per original game-platform record (one per region). This **generates reporting rows**, not new games. Keep missing regional sales as missing so that missing is not confused with zero.

In [18]:
regions = {
    "na_sales": "North America",
    "jp_sales": "Japan",
    "pal_sales": "Europe and Africa",
    "other_sales": "Other"
}

regional_sales = games.melt(
    id_vars=["record_id", "title", "platform", "genre", "publisher", "release_year"],
    value_vars=list(regions),
    var_name="region",
    value_name="sales_millions"
)
regional_sales["region"] = regional_sales["region"].map(regions)
regional_sales["has_sales"] = regional_sales["sales_millions"].notna()

assert len(regional_sales) == 4 * len(games)
print("Regional rows:", len(regional_sales))
print(regional_sales.groupby("region")["sales_millions"].agg(
    observations="count", observed_sales=lambda s: s.sum(min_count=1)
))

Regional rows: 256064
                   observations  observed_sales
region                                         
Europe and Africa         12824         1916.83
Japan                      6726          687.94
North America             12637         3345.52
Other                     15128          651.12


## 5. Format Data — Prepare final tables

Put identifying columns first, remove temporary join keys, retain missing measurements, and save pandas Pickles for Dash. This only changes representation; it should not change sales totals or the number of game records.

In [19]:
enriched = enriched.drop(columns=["developer_key", "publisher_key"])

first_columns = [
    "record_id", "title", "platform", "genre", "publisher", "developer",
    "release_date", "release_year", "total_sales", "na_sales",
    "jp_sales", "pal_sales", "other_sales"
]
enriched = enriched[first_columns + [c for c in enriched.columns if c not in first_columns]]

# Checks before saving: no game rows lost/duplicated and missing sales preserved.
assert len(enriched) == len(raw["charts"])
assert enriched["record_id"].is_unique
assert enriched["total_sales"].isna().sum() == raw["charts"]["total_sales"].isna().sum()
assert enriched["total_sales"].sum() == raw["charts"]["total_sales"].sum()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
enriched.to_pickle(OUTPUT_DIR / "games_prepared.pkl")
regional_sales.to_pickle(OUTPUT_DIR / "regional_sales_long.pkl")

print("Saved:")
print(OUTPUT_DIR / "games_prepared.pkl", enriched.shape)
print(OUTPUT_DIR / "regional_sales_long.pkl", regional_sales.shape)

Saved:
..\..\data\prepared\games_prepared.pkl (64016, 27)
..\..\data\prepared\regional_sales_long.pkl (256064, 9)


## 6. Final checks and report notes

Record the *actual output* below in your Phase 3 Markdown report. Do not assume the existing Phase 2 numbers are unchanged after parsing or merging.

Checklist:
1. Row counts before/after cleaning and joins.
2. Missing sales count preserved; number of failed conversions.
3. Repeated title/platform combinations **flagged**, not deleted.
4. Company and country lookup match rates; unmatched names retained.
5. Long-format table contains four rows per source record.
6. Paths, schemas, and intended uses of both saved `.pkl` files.

**Next phase:** The Dash app loads the prepared Pickles instead of repeating all cleaning steps on every run.

In [20]:
print("Original game rows:", len(raw["charts"]))
print("Prepared game rows:", len(enriched))
print("Regional rows:", len(regional_sales))
print("Game table columns:", enriched.columns.tolist())
print("Regional table columns:", regional_sales.columns.tolist())
print("Sales coverage:", round(enriched["total_sales"].notna().mean() * 100, 2), "%")

Original game rows: 64016
Prepared game rows: 64016
Regional rows: 256064
Game table columns: ['record_id', 'title', 'platform', 'genre', 'publisher', 'developer', 'release_date', 'release_year', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales', 'img', 'critic_score', 'last_update', 'has_total_sales', 'developer_city', 'developer_country', 'publisher_city', 'publisher_country', 'developer_iso3', 'developer_latitude', 'developer_longitude', 'publisher_iso3', 'publisher_latitude', 'publisher_longitude']
Regional table columns: ['record_id', 'title', 'platform', 'genre', 'publisher', 'release_year', 'region', 'sales_millions', 'has_sales']
Sales coverage: 29.56 %


In [21]:
# read games_prepared.pkl and give head(10)
games_prepared_pkl = pd.read_pickle(OUTPUT_DIR / "games_prepared.pkl")
games_prepared_pkl.head(10)

,record_id,title,platform,genre,publisher,developer,release_date,release_year,total_sales,na_sales,...,developer_city,developer_country,publisher_city,publisher_country,developer_iso3,developer_latitude,developer_longitude,publisher_iso3,publisher_latitude,publisher_longitude
0,1,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,2013-09-17,2013,20.32,6.37,...,NaN,NaN,"New York, New York",United States,<NA>,<NA>,<NA>,USA,38.0,-97.0
1,2,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,2014-11-18,2014,19.39,6.06,...,NaN,NaN,"New York, New York",United States,<NA>,<NA>,<NA>,USA,38.0,-97.0
2,3,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,2002-10-28,2002,16.15,8.41,...,NaN,NaN,"New York, New York",United States,<NA>,<NA>,<NA>,USA,38.0,-97.0
3,4,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,2013-09-17,2013,15.86,9.06,...,NaN,NaN,"New York, New York",United States,<NA>,<NA>,<NA>,USA,38.0,-97.0
4,5,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,2015-11-06,2015,15.09,6.18,...,Santa Monica,United States,"Santa Monica, California",United States,USA,38.0,-97.0,USA,38.0,-97.0
5,6,Call of Duty: Modern Warfare 3,X360,Shooter,Activision,Infinity Ward,2011-11-08,2011,14.82,9.07,...,Woodland Hills,United States,"Santa Monica, California",United States,USA,38.0,-97.0,USA,38.0,-97.0
6,7,Call of Duty: Black Ops,X360,Shooter,Activision,Treyarch,2010-11-09,2010,14.74,9.76,...,Santa Monica,United States,"Santa Monica, California",United States,USA,38.0,-97.0,USA,38.0,-97.0
7,8,Red Dead Redemption 2,PS4,Action-Adventure,Rockstar Games,Rockstar Games,2018-10-26,2018,13.94,5.26,...,NaN,NaN,"New York, New York",United States,<NA>,<NA>,<NA>,USA,38.0,-97.0
8,9,Call of Duty: Black Ops II,X360,Shooter,Activision,Treyarch,2012-11-13,2012,13.86,8.27,...,Santa Monica,United States,"Santa Monica, California",United States,USA,38.0,-97.0,USA,38.0,-97.0
9,10,Call of Duty: Black Ops II,PS3,Shooter,Activision,Treyarch,2012-11-13,2012,13.80,4.99,...,Santa Monica,United States,"Santa Monica, California",United States,USA,38.0,-97.0,USA,38.0,-97.0
